# Qiskit Fall Fest 2026 — Phase 1: Problem S3
## Controlled Architecture Benchmarking: Noise Accumulation in Trotter Simulation
**Track:** Quantum Simulation  
**Problem Statement:** S3 — Noise in Trotter Simulation  
**Target Processors:** Processor A (5-Qubit Linear Chain) vs. Processor B (9-Qubit Heavy-Hex Lattice)  
**Benchmark Scope:** Open-Chain TFIM Baseline ($J_{\text{cross}}=0.0$) and Periodic Routing Extension ($J_{\text{cross}}=0.5$)  

---
### Abstract & Scientific Objectives
Simulating non-commuting Hamiltonians on near-term noisy quantum hardware requires discretizing continuous time evolution via product formulas (e.g., the first-order Trotter-Suzuki decomposition). This introduces a fundamental physical trade-off:
1. **Discretization Error:** Fine time-slicing (higher step count $n$) mathematically suppresses product-formula error $\mathcal{O}(\Delta t^2)$.
2. **Hardware Noise Accumulation:** Each Trotter step introduces additional two-qubit entangling gates ($CX$), compounding depolarizing and readout errors.
3. **Coupling Topology Overhead:** When simulating non-local interactions (such as the periodic ring boundary), hardware with constrained connectivity incurs significant routing overhead (SWAP networks), multiplying the physical gate depth.

This walkthrough implements a rigorously controlled, reproducible benchmark investigating both effects on **Processor A (5Q Line)** and **Processor B (9Q Heavy-Hex)**.

### 1. Environment Setup & Software Provenance
We verify all dependencies, Qiskit versions, and configure reproducible random seeds.

In [1]:
import sys
import json
from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import numpy as np
import scipy.linalg
import pandas as pd
import matplotlib.pyplot as plt
try:
    import importlib
    _disp_mod = importlib.import_module('IPython.display')
    Image = getattr(_disp_mod, 'Image')
    display = getattr(_disp_mod, 'display')
except Exception:
    class Image:
        def __init__(self, filename=None, *args, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"<Image: {self.filename}>"
    def display(*args, **kwargs):
        for a in args:
            print(a)

import qiskit
import qiskit_aer
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

from src.processors import load_processor_config, build_coupling_map, build_noise_model
from src.problem import TrotterSimulationModel
from src.transpilation import compile_for_processor, extract_circuit_metrics
from src.metrics import (
    compute_single_site_z_from_counts,
    compute_magnetization_from_counts,
    compute_tvd,
    compute_classical_distribution_fidelity
)
from src.visualization import (
    plot_processor_topology,
    plot_logical_vs_transpiled,
    plot_ab_performance,
    plot_architecture_tradeoffs
)

print(f"Python Version:     {sys.version.split()[0]}")
print(f"Qiskit Version:     {qiskit.__version__}")
print(f"Qiskit Aer Version: {qiskit_aer.__version__}")
print(f"NumPy Version:      {np.__version__}")
print(f"SciPy Version:      {scipy.__version__}")
print("Environment verified successfully.")

Python Version:     3.11.9
Qiskit Version:     2.5.2
Qiskit Aer Version: 0.17.2
NumPy Version:      2.4.6
SciPy Version:      1.17.1
Environment verified successfully.


### 2. Physical Model & Observables Formulation
We consider a 4-spin Transverse-Field Ising Model (TFIM) governed by the Hamiltonian:
$$\hat{H} = -J \sum_{i=0}^{N-2} Z_i Z_{i+1} - J_{\text{cross}} Z_0 Z_{N-1} - h \sum_{i=0}^{N-1} X_i$$
with parameters $J = 1.0$, $h = 0.8$, total evolution time $T = 1.0$, and staggered initial Néel state $|\psi(0)\rangle = |1010\rangle$.

- **Primary Baseline:** Open-chain boundary condition ($J_{\text{cross}} = 0.0$).
- **Architectural Extension:** Periodic boundary condition ($J_{\text{cross}} = 0.5$) creating a ring topology to test routing stress.

#### Observables:
1. **Primary Observable:** Single-site expectation value $\langle Z_0(t) \rangle$. At $t=0$, $\langle Z_0(0) \rangle = +1.0$. At $T=1.0$, $\langle Z_0(1.0) \rangle$ exhibits clear Trotter convergence toward its exact numerical continuum value.
2. **Secondary Physical Diagnostic:** Total average magnetization $\langle M_z(t) \rangle = \frac{1}{N} \sum_{i=0}^{N-1} \langle Z_i(t) \rangle$. Under spatial reflection anti-symmetry $\mathcal{P} \hat{H} \mathcal{P}^\dagger = \hat{H}$ and initial state $|1010\rangle$, $\langle Z_0(t) \rangle = -\langle Z_3(t) \rangle$ and $\langle Z_1(t) \rangle = -\langle Z_2(t) \rangle$, rendering exact $\langle M_z(t) \rangle \equiv 0.0$ identically for all $t$.

In [2]:
num_spins = 4
T = 1.0

# Initialize primary baseline model (Open chain)
model_open = TrotterSimulationModel(num_spins=num_spins, J=1.0, h=0.8, boundary="open", initial_state_str="1010")
z0_exact_open = model_open.compute_exact_single_site_z(T, site=0)
mz_exact_open = model_open.compute_exact_magnetization(T)

# Initialize architectural extension model (Periodic ring)
model_per = TrotterSimulationModel(num_spins=num_spins, J=1.0, h=0.8, boundary="periodic", initial_state_str="1010")
z0_exact_per = model_per.compute_exact_single_site_z(T, site=0)
mz_exact_per = model_per.compute_exact_magnetization(T)

print("Exact Continuum Reference Values at T = 1.0 (via scipy.linalg.expm):")
print(f"  [Open Chain Baseline]    <Z_0(1.0)> = {z0_exact_open:+.6f} | <M_z(1.0)> = {mz_exact_open:+.6f}")
print(f"  [Periodic Ring Extension] <Z_0(1.0)> = {z0_exact_per:+.6f} | <M_z(1.0)> = {mz_exact_per:+.6f}")

# Verify reflection anti-symmetry numerically
psi_open = model_open.compute_exact_statevector(T)
from qiskit.quantum_info import SparsePauliOp
z_vals = [psi_open.expectation_value(SparsePauliOp.from_list([(''.join(['Z' if k == i else 'I' for k in reversed(range(4))]), 1.0)])).real for i in range(4)]
print(f"\nSingle-site individual expectations: <Z_0>={z_vals[0]:.4f}, <Z_1>={z_vals[1]:.4f}, <Z_2>={z_vals[2]:.4f}, <Z_3>={z_vals[3]:.4f}")
print(f"Pairwise sums: <Z_0> + <Z_3> = {z_vals[0] + z_vals[3]:.1e}, <Z_1> + <Z_2> = {z_vals[1] + z_vals[2]:.1e}")

Exact Continuum Reference Values at T = 1.0 (via scipy.linalg.expm):
  [Open Chain Baseline]    <Z_0(1.0)> = +0.257957 | <M_z(1.0)> = -0.000000
  [Periodic Ring Extension] <Z_0(1.0)> = +0.421887 | <M_z(1.0)> = +0.000000

Single-site individual expectations: <Z_0>=0.2580, <Z_1>=-0.5742, <Z_2>=0.5742, <Z_3>=-0.2580
Pairwise sums: <Z_0> + <Z_3> = -8.3e-16, <Z_1> + <Z_2> = 2.2e-16


### 3. Hardware Processor Topologies & Noise Configuration
We configure two hardware architectures conforming to official Master Guide specifications:
- **Processor A (5Q Line):** 5 physical qubits arranged in a linear chain `0 - 1 - 2 - 3 - 4` with designated I/O ports `[0, 4]`.
- **Processor B (9Q Heavy-Hex):** 9 physical qubits with 12 undirected edges (24 directed pairs) embedding a heavy-hex hexagonal unit cell with I/O ports `[0, 8]`.

#### Noise Model Parameters:
- Single-qubit depolarizing rate: $p_1 = 0.001$
- Two-qubit depolarizing rate: $p_2 = 0.005$ per CX
- Readout error: Symmetric $1\%$ bit-flip matrix $\begin{pmatrix} 0.99 & 0.01 \\ 0.01 & 0.99 \end{pmatrix}$

In [3]:
proc_a_cfg = load_processor_config("processors/processor_A.json")
proc_b_cfg = load_processor_config("processors/processor_B.json")

cmap_a = build_coupling_map(proc_a_cfg)
cmap_b = build_coupling_map(proc_b_cfg)

noise_a = build_noise_model(proc_a_cfg)
noise_b = build_noise_model(proc_b_cfg)

print(f"Processor A: {proc_a_cfg['num_qubits']} qubits, {len(proc_a_cfg['coupling_map'])} directed edges, basis={proc_a_cfg['basis_gates']}")
print(f"Processor B: {proc_b_cfg['num_qubits']} qubits, {len(proc_b_cfg['coupling_map'])} directed edges, basis={proc_b_cfg['basis_gates']}")

# Display coupling topologies
display(Image(filename="results/figures/processor_A_graph.png"))
display(Image(filename="results/figures/processor_B_graph.png"))

Processor A: 5 qubits, 8 directed edges, basis=['rz', 'sx', 'x', 'cx']
Processor B: 9 qubits, 24 directed edges, basis=['rz', 'sx', 'x', 'cx']
<Image: results/figures/processor_A_graph.png>
<Image: results/figures/processor_B_graph.png>


### 4. Circuit Synthesis & Architectural Transpilation Analysis
We examine the transpilation behavior for both the open-chain baseline and the periodic ring extension at $n=4$ Trotter steps.

Notice the crucial architectural distinction:
- For the **open-chain baseline**, both processors directly embed the 4-qubit line without any routing overhead ($\Delta CX = 0$).
- For the **periodic ring extension**, the non-local term $Z_0 Z_3$ closes the loop. Processor A must insert 14 SWAP operations (42 excess CX gates) to shuttle qubit states along the linear chain. Processor B, having a natural cycle `(1, 2, 5, 6)` in its heavy-hex subgraph, embeds the 4-ring directly with **zero routing overhead**!

In [4]:
# Transpilation comparison on Periodic Ring at n=4
qc_per = model_per.build_trotter_circuit(t=1.0, num_steps=4, measure=True)
met_logical = extract_circuit_metrics(qc_per)

tc_a, met_a = compile_for_processor(qc_per, cmap_a, basis_gates=proc_a_cfg["basis_gates"], optimization_level=1, seed_transpiler=42)
tc_b, met_b = compile_for_processor(qc_per, cmap_b, basis_gates=proc_b_cfg["basis_gates"], optimization_level=1, seed_transpiler=42)

transpile_df = pd.DataFrame({
    "Metric": ["Total Processor Qubits", "Active Qubits Used", "Circuit Depth", "Two-Qubit Depth", "Two-Qubit CX Gates", "Routing CX Overhead (ΔCX)", "Estimated SWAPs (ΔCX/3)"],
    "Logical Circuit": [4, 4, met_logical["depth"], met_logical["two_qubit_depth"], met_logical["two_qubit_ops"], 0, 0],
    "Processor A (5Q Line)": [met_a["total_processor_qubits"], met_a["active_physical_qubits"], met_a["depth"], met_a["two_qubit_depth"], met_a["two_qubit_ops"], met_a["two_qubit_ops"] - met_logical["two_qubit_ops"], (met_a["two_qubit_ops"] - met_logical["two_qubit_ops"]) // 3],
    "Processor B (9Q Heavy-Hex)": [met_b["total_processor_qubits"], met_b["active_physical_qubits"], met_b["depth"], met_b["two_qubit_depth"], met_b["two_qubit_ops"], met_b["two_qubit_ops"] - met_logical["two_qubit_ops"], (met_b["two_qubit_ops"] - met_logical["two_qubit_ops"]) // 3]
})
display(transpile_df)

                      Metric  ...  Processor B (9Q Heavy-Hex)
0     Total Processor Qubits  ...                           9
1         Active Qubits Used  ...                           4
2              Circuit Depth  ...                          70
3            Two-Qubit Depth  ...                          32
4         Two-Qubit CX Gates  ...                          32
5  Routing CX Overhead (ΔCX)  ...                           0
6    Estimated SWAPs (ΔCX/3)  ...                           0

[7 rows x 4 columns]


### 5. Quantitative Benchmark Results & Trade-off Tables
We load and inspect the benchmark results produced across the full step sweep $n \in [1, 2, 3, 4, 5, 6, 8, 10]$ under 4096 measurement shots.

In [5]:
df_final = pd.read_csv("results/tables/final_comparison.csv")
print("=== Primary Open-Chain Baseline Summary ===")
display(df_final)

df_periodic = pd.read_csv("results/tables/periodic_extension_comparison.csv")
print("\n=== Periodic Ring Extension Routing Summary ===")
display(df_periodic[["Trotter_Steps_n", "Logical_2Q_Gates", "ProcA_2Q_Gates", "ProcB_2Q_Gates", "ProcA_Routing_CX_Overhead", "ProcB_Routing_CX_Overhead", "2Q_Gate_Delta_B_minus_A"]])

=== Primary Open-Chain Baseline Summary ===
                                               Metric  ...                        Difference (B - A)
0                      Total Processor Qubit Capacity  ...                        +4 qubits capacity
1                         Active Physical Qubits Used  ...           0 (Both utilize 4 active spins)
2                                   Coupling Topology  ...  Heavy-hex 12-edge lattice vs simple line
3                           Total Circuit Depth (n=4)  ...                                         0
4                       Two-Qubit Circuit Depth (n=4)  ...                                         0
5                            Two-Qubit CX Gates (n=4)  ...                                         0
6                           Routing CX Overhead (n=4)  ...                                         0
7             Estimated Equivalent SWAPs (ΔCX/3, n=4)  ...                                         0
8                    Explicit SWAP Instructions

### 6. Publication-Grade Visualizations & Dashboard
We examine the graphical findings across three complementary perspectives:
1. **Circuit Cost Scaling:** Circuit depth and two-qubit gate counts vs. Trotter step count $n$.
2. **Simulation Fidelity Trade-off:** Algorithmic pure-state fidelity $F_{\text{state}}$ vs. Noisy measurement distribution overlap $F_c$.
3. **Architecture Trade-offs Dashboard:** 4-panel analysis detailing TVD, routing CX overhead, observable error $|\langle Z_0 \rangle_{\text{measured}} - \langle Z_0 \rangle_{\text{exact}}|$, and 2Q gate accumulation.

In [6]:
print("Figure 1: Logical vs. Transpiled Circuit Scaling")
display(Image(filename="results/figures/logical_vs_transpiled.png"))

print("\nFigure 2: Trotter Simulation Fidelity (Discretization vs. Hardware Noise)")
display(Image(filename="results/figures/AB_performance.png"))

print("\nFigure 3: Comprehensive Architecture Trade-offs Dashboard")
display(Image(filename="results/figures/architecture_tradeoffs.png"))

Figure 1: Logical vs. Transpiled Circuit Scaling
<Image: results/figures/logical_vs_transpiled.png>

Figure 2: Trotter Simulation Fidelity (Discretization vs. Hardware Noise)
<Image: results/figures/AB_performance.png>

Figure 3: Comprehensive Architecture Trade-offs Dashboard
<Image: results/figures/architecture_tradeoffs.png>


### 7. Key Scientific Findings & Conclusions

#### 1. Algorithmic Error vs. Hardware Noise Trade-off:
- Ideal pure-state Trotter fidelity increases monotonically with step count $n$: from $0.6272$ at $n=1$ to $0.9648$ at $n=4$ and $0.9949$ at $n=10$.
- However, on noisy hardware, gate errors compound with circuit depth. The classical distribution overlap $F_c$ peaks at low step counts ($n^* = 1$ to $4$) before decaying as depolarizing noise randomizes output bitstrings.

#### 2. Architectural Advantage of Processor B Heavy-Hex Lattice:
- When non-local interactions occur (e.g. periodic boundary conditions), Processor B achieves a **56.8% reduction in two-qubit operations at $n=4$** (32 CX vs 74 CX on Processor A) and **58.8% reduction at $n=10$** (80 CX vs 194 CX on Processor A).
- This routing efficiency directly mitigates coherent error and two-qubit depolarizing noise accumulation.

#### 3. Observable Symmetry Validation:
- The primary observable $\langle Z_0(t) \rangle$ reliably tracks Trotter convergence, achieving an error $|\langle Z_0 \rangle - \langle Z_0 \rangle_{\text{exact}}| = 0.027$ at $n=4$.
- The secondary diagnostic $\langle M_z(t) \rangle$ confirms exact theoretical reflection symmetry by vanishing identically ($0.000000$) across all time steps.